# Amazon ML Challenge 2026 — Business Entity Resolution
**Single Colab Notebook — Run Top to Bottom**

## Architecture Summary
| Stage | Strategy | Why |
|---|---|---|
| Blocking | Token + Sorted-neighborhood + FAISS ANN | Union maximises recall ceiling |
| Matcher | LightGBM on ~24 language-agnostic features | Best P/R tradeoff on tabular; generalises to unseen France |
| Threshold | Macro F0.5 sweep on held-out val | Matches exact scoring function |
| Post-process | 1-to-1 dedup + graph pruning | Pure precision gains, zero recall cost |

**Evaluation metric**: Macro-averaged F₀.5 (precision weighted 2× over recall)

---

## Cell 1 — Install Dependencies
Run once per Colab session. Colab Pro+ with GPU recommended for embedding stage.

In [ ]:
# Install all required packages
# NOTE: restart runtime AFTER this cell completes

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

packages = [
    "polars==0.20.31",
    "pyarrow==16.1.0",
    "rapidfuzz==3.9.3",
    "jellyfish==1.0.4",
    "lightgbm==4.3.0",
    "sentence-transformers==3.0.1",
    "faiss-cpu==1.8.0",        # swap to faiss-gpu if on A100/T4
    "scikit-learn==1.5.0",
    "tqdm==4.66.4",
    "huggingface_hub==0.23.4",
]

for pkg in packages:
    print(f"Installing {pkg} ...")
    pip_install(pkg)

print("\nAll packages installed. RESTART RUNTIME now, then continue from Cell 2.")

## Cell 2 — Mount Google Drive & Set Paths
After restarting: run from here.

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
except ImportError:
    print("Not running in Colab — skipping drive mount.")

import os, sys

# ── Auto-detect src/ directory (Works on SageMaker, Local, and Colab) ────────
if os.path.exists("src"):
    SRC_DIR = os.path.abspath("src")
elif os.path.exists("../src"):
    SRC_DIR = os.path.abspath("../src")
else:
    curr = os.path.abspath(os.getcwd())
    found = None
    for _ in range(5):
        cand = os.path.join(curr, "code", "business_entity_resolution", "src")
        if os.path.exists(cand):
            found = cand
            break
        cand2 = os.path.join(curr, "src")
        if os.path.exists(cand2):
            found = cand2
            break
        curr = os.path.dirname(curr)
    SRC_DIR = found or os.path.abspath("src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"SRC_DIR added to sys.path: {SRC_DIR}")

# Verify config can be imported
import config
import importlib
importlib.reload(config)

print(f"DATA_ROOT     = {config.DATA_ROOT}")
print(f"TRAIN_DIR     = {config.TRAIN_DIR}")
print(f"ARTIFACTS_DIR = {config.ARTIFACTS_DIR}")
print(f"OUTPUT_DIR    = {config.OUTPUT_DIR}")

os.makedirs(config.ARTIFACTS_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)


## Cell 3 — Load Training Data
Loads all 3 train source TSVs and the ground truth.
Uses Polars for memory-efficient reading (~1.1 GB total uncompressed).

In [ ]:
# Loads train_source1, train_source2, train_source3, train_ground_truth

import pipeline

print("Loading training data ...")
s1, s2, s3, gt = pipeline.stage_load_train()

print(f"\nS1 shape : {s1.shape}")
print(f"S2 shape : {s2.shape}")
print(f"S3 shape : {s3.shape}")
print(f"GT shape : {gt.shape}")
print(s1.head(3))

## Cell 4 — Add Normalised Text Columns
Applies NFKC + lowercase + legal-suffix stripping + token sort to name & address.
Token-sorted normalisation makes "Acme Robotics" == "Robotics Acme".

In [ ]:
# Adds norm_name and norm_addr columns to each source DataFrame.
# This takes ~5-10 minutes per source on Colab CPU. Cache if restarting often.

print("Normalising S1 ...")
s1 = pipeline.stage_add_norm_cols(s1)

print("Normalising S2 ...")
s2 = pipeline.stage_add_norm_cols(s2)

print("Normalising S3 ...")
s3 = pipeline.stage_add_norm_cols(s3)

print("\nSample normalised names:")
print(s1.select(["entity_id", "business_name", "norm_name"]).head(5))

## Cell 5 — Encode Train Sources with Multilingual Sentence Encoder
Model: `paraphrase-multilingual-MiniLM-L12-v2` (MIT license, 117M params)

Handles Hindi-transliteration (present in train data) and French (unseen at train time).
Embeddings are L2-normalised → dot product == cosine similarity.

**Runtime**: ~1-2 hours on Colab T4 for all 3 train sources (~12.5M rows total).
Saved to ARTIFACTS_DIR — skip next time by using cached .npy files.

In [ ]:
# Encode train sources and cache embeddings.
# Set force=True only if you want to re-encode (e.g. after changing the model).

FORCE_RECOMPUTE = False

print("Encoding train sources (GPU accelerated if available) ...")
e1, e2, e3 = pipeline.stage_encode_train(s1, s2, s3, force=FORCE_RECOMPUTE)

print(f"\nEmbedding shapes:  S1={e1.shape}  S2={e2.shape}  S3={e3.shape}")
print(f"Dtype: {e1.dtype}")

## Cell 6 — Generate Blocking Candidates (Train)
Combines 3 strategies:
  A. Token blocking (name/address prefix keys)
  B. Sorted-neighborhood (window over sorted norm_name)
  C. FAISS ANN top-30 per S1 entity

Country partitioning applied first (zero recall loss on training GT).
Target: recall >= 0.99, reduction ratio >= 0.999

In [ ]:
# This generates all candidate pairs from training data.
# Output: dict { s1_id : set of candidate s2/s3 ids }

candidates = pipeline.stage_blocking_train(s1, s2, s3, e1, e2, e3)

# Quick stats
import numpy as np
cand_sizes = [len(v) for v in candidates.values()]
print(f"\nBlocking candidate stats:")
print(f"  S1 entities with >= 1 candidate: {sum(1 for v in candidates.values() if v):,}")
print(f"  Mean candidates per S1          : {np.mean(cand_sizes):.1f}")
print(f"  P95 candidates per S1           : {np.percentile(cand_sizes, 95):.0f}")
print(f"  Max candidates per S1           : {max(cand_sizes):,}")
print(f"  Total candidate pairs           : {sum(cand_sizes):,}")

## Cell 7 — Measure Blocking Recall
This is the hard ceiling on your final F0.5 score.
Any GT pair not captured here CANNOT be recovered downstream.
Target: >= 0.99 (ideally >= 0.995)

In [ ]:
# Compute blocking recall against full training ground truth.

gt_dict = pipeline.gt_to_dict(gt)

recall = pipeline.stage_blocking_recall(candidates, gt)

print(f"\nBlocking recall = {recall:.4f}")
if recall < 0.97:
    print("ACTION NEEDED: Increase ANN_TOP_K or SNM_WINDOW in config.py")
elif recall < 0.99:
    print("GOOD: Consider increasing ANN_TOP_K slightly for more headroom.")
else:
    print("EXCELLENT: Blocking recall meets target.")

## Cell 8 — Train / Validation Split
Group-split by source1_entity_id (no leakage).
Stratified by match-count bucket (0,1,2,3,4,5+).
80% train / 20% validation.

In [ ]:
# Returns sets of S1 entity_ids for train and val partitions.

train_s1_ids, val_s1_ids = pipeline.stage_train_val_split(gt)

print(f"Train S1 entities : {len(train_s1_ids):,}")
print(f"Val   S1 entities : {len(val_s1_ids):,}")

## Cell 9 — Build Record Lookups (Train)
Creates a single merged dict { entity_id -> record } for all train S1/S2/S3.
Attaches embedding vectors so they're available as features without re-encoding.

In [ ]:
# Merge lookups for all train entities. Takes ~5 min on CPU.

lookup_all = pipeline.stage_build_lookups(s1, s2, s3, e1, e2, e3)
print(f"lookup_all entries: {len(lookup_all):,}")

## Cell 10 — Build Training Pair Dataset
Positives: all GT matches found in candidates.
Hard negatives: same-block non-matches, downsampled 8:1 to positives.

In [ ]:
# Returns X_train (DataFrame of ~24 features) and y_train (0/1 labels).

X_train, y_train = pipeline.stage_build_training_pairs(
    train_s1_ids, candidates, gt_dict, lookup_all
)

print(f"\nX_train shape    : {X_train.shape}")
print(f"Positive pairs   : {y_train.sum():,}")
print(f"Negative pairs   : {(y_train == 0).sum():,}")
print(f"Positive rate    : {y_train.mean():.4f}")
print("\nFeature sample:")
print(X_train.head(3))

## Cell 11 — Train LightGBM Matcher
Binary classifier on ~24 language-agnostic similarity features.
Early stopping on val log-loss (separate small val sample, not the full sweep set).
scale_pos_weight < 1.0 biases toward precision (matches F0.5 objective).

In [ ]:
# Trains and saves the model to ARTIFACTS_DIR/lgbm_matcher.pkl

model = pipeline.stage_train_model(
    X_train, y_train, val_s1_ids, candidates, gt_dict, lookup_all
)

print("\nTop-10 feature importances:")
import pandas as pd
from features import FEATURE_NAMES
fi = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
print(fi.head(10).to_string())

## Cell 12 — Threshold Sweep (Macro F0.5)
Sweeps probability thresholds from 0.30 to 0.95 in 0.01 steps.
Picks the threshold that maximises MACRO-averaged F0.5 on the val split.
This is NOT the same as optimising micro F0.5 — macro is what the challenge uses.

In [ ]:
# Returns the optimal threshold (typically 0.55 - 0.80 due to precision weighting).

threshold = pipeline.stage_threshold_sweep(
    model, val_s1_ids, candidates, gt_dict, lookup_all
)

print(f"\nOptimal threshold: {threshold:.4f}")
print("This threshold will be used for test set inference.")

## Cell 13 — Full Validation Evaluation Report
Computes macro F0.5, precision, recall, and singleton accuracy at the optimal threshold.
This is your realistic estimate of the leaderboard score.

In [ ]:
# Full evaluation on the held-out 20% val set.

results = pipeline.stage_evaluate(
    model, threshold, val_s1_ids, candidates, gt_dict, lookup_all
)

print(f"\nExpected leaderboard F0.5 ≈ {results['macro_f05']:.5f}")

## Cell 14 — Load Test Data & Encode
Load test source files. Encode with the same multilingual model.
Embeddings cached to ARTIFACTS_DIR — safe to restart without re-encoding.

In [ ]:
# Load test sources
print("Loading test data ...")
ts1, ts2, ts3 = pipeline.stage_load_test()

# Normalise
print("Normalising test sources ...")
ts1 = pipeline.stage_add_norm_cols(ts1)
ts2 = pipeline.stage_add_norm_cols(ts2)
ts3 = pipeline.stage_add_norm_cols(ts3)

# Encode (loads from cache if available)
print("Encoding test sources ...")
te1, te2, te3 = pipeline.stage_encode_test(ts1, ts2, ts3)

print(f"\nTest embedding shapes: S1={te1.shape}  S2={te2.shape}  S3={te3.shape}")

## Cell 15 — Generate Test Blocking Candidates
Same blocking pipeline on test data.
France records (15% of test) handled automatically by the multilingual encoder.

In [ ]:
# Generate candidate pairs for the test set.

test_candidates = pipeline.stage_blocking_test(ts1, ts2, ts3, te1, te2, te3)

cand_sizes_test = [len(v) for v in test_candidates.values()]
print(f"\nTest blocking stats:")
print(f"  S1 with >= 1 candidate: {sum(1 for v in test_candidates.values() if v):,}")
print(f"  Mean candidates / S1  : {np.mean(cand_sizes_test):.1f}")
print(f"  Total candidate pairs  : {sum(cand_sizes_test):,}")

## Cell 16 — Build Test Lookup Dict

In [ ]:
# Merge test entity lookups with embeddings attached.

test_lookup_all = pipeline.stage_build_test_lookups(ts1, ts2, ts3, te1, te2, te3)
print(f"test_lookup_all entries: {len(test_lookup_all):,}")

## Cell 17 — Test Set Inference + Post-processing + Write Outputs
1. Score all test candidate pairs with LightGBM (batched, 100K pairs/batch).
2. Apply optimal threshold from Cell 12.
3. One-to-one dedup (EDA-backed constraint: each S2/S3 maps to at most 1 S1).
4. Graph consistency pruning (removes internally inconsistent match sets).
5. Write `matching_results.tsv` and `candidate_pairs.tsv` to output/.

In [ ]:
# Full test inference pipeline. Runtime: 20-60 min depending on candidate volume.

predictions = pipeline.stage_inference(
    model, threshold, ts1, test_candidates, test_lookup_all
)

print(f"\nTotal S1 rows in output: {len(predictions):,}")
print(f"S1 with matches        : {sum(1 for v in predictions.values() if v):,}")
print(f"Singletons (empty)     : {sum(1 for v in predictions.values() if not v):,}")
print(f"Total link predictions : {sum(len(v) for v in predictions.values()):,}")

## Cell 18 — Validate Submission
Runs the stdlib-only validator. Must print PASS before you upload.
Checks: all S1 rows present, no duplicates, all matched IDs in S2/S3,
every matched ID in candidate set.

In [ ]:
# If this cell raises an error, DO NOT SUBMIT. Fix the issue first.

pipeline.stage_validate_submission()

## Cell 19 — Preview Output Files
Sanity-check the output before uploading to the portal.

In [ ]:
import polars as pl
import config

# Preview matching_results.tsv
print("=== matching_results.tsv (first 10 rows) ===")
matching_preview = pl.read_csv(
    config.MATCHING_OUT, separator="\t",
    null_values=[""]
)
print(matching_preview.head(10))

print(f"\nTotal rows: {len(matching_preview):,}")

# Preview candidate_pairs.tsv
print("\n=== candidate_pairs.tsv (first 5 rows) ===")
cand_preview = pl.read_csv(
    config.CANDIDATE_OUT, separator="\t",
    null_values=[""]
)
print(cand_preview.head(5))

## Cell 20 — Download Output for Submission (Colab only)
Downloads `matching_results.tsv` to your local machine.
Upload this file to the Portal leaderboard.

In [ ]:
try:
    from google.colab import files
    print("Downloading matching_results.tsv ...")
    files.download(config.MATCHING_OUT)
    print("Done. Upload matching_results.tsv to the Amazon ML Challenge portal.")
except ImportError:
    print(f"Not in Colab. Find your output at:\n  {config.MATCHING_OUT}")

---
## Debugging Checklist

| Problem | Likely Cause | Fix |
|---|---|---|
| Blocking recall < 0.97 | ANN_TOP_K too small | Increase to 50 in config.py |
| Val F0.5 < 0.85 | Threshold at wrong point | Check threshold sweep output |
| Validator FAILS | Duplicate IDs / missing S1 rows | Run Cell 18 and read error |
| OOM during encoding | Batch size too large | Set EMBED_BATCH_SIZE=256 in config.py |
| French records all singletons | Embedding quality | Try `multilingual-e5-small` |

## SageMaker Migration
1. Set `DATA_ROOT` env var or edit `config.py` directly.
2. Run `pip install -r requirements.txt`.
3. Run `python src/pipeline.py` (add a `__main__` guard calling stages in order).
4. Retrieve outputs from `output/`.